# CPSC 452 Final Project

## Neural Conservation Discovery + Sparse Equation Recovery

This notebook runs the **core pipeline** end-to-end:
- Load (or optionally regenerate) projectile + pendulum trajectories
- Scale data (min-max for projectile, standardize for pendulum) for neural training
- Train CDN and validate vs analytical energy
- Run sparse equation recovery via `scripts/run_equation_discovery.py` (Polynomial + L1/LASSO + SINDy STLSQ; optional PySR)


## Setup

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# Build absolute project paths (works in Jupyter or nbconvert)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "figures")

# Ensure imports work when running from notebooks/
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Models dir:", MODELS_DIR)
print("Figures dir:", FIGURES_DIR)
print("Torch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## Day 1 — Data generation (optional) + load

In [ ]:
from src.data_generation.projectile import generate_projectile_data
from src.data_generation.pendulum import generate_pendulum_data

# Toggle to regenerate .npy files (otherwise we just load existing)
REGENERATE_DATA = False

proj_path = os.path.join(DATA_DIR, "projectile", "trajectories.npy")
pend_path = os.path.join(DATA_DIR, "pendulum", "trajectories.npy")

if REGENERATE_DATA:
    os.makedirs(os.path.join(DATA_DIR, "projectile"), exist_ok=True)
    os.makedirs(os.path.join(DATA_DIR, "pendulum"), exist_ok=True)

    proj = generate_projectile_data()
    np.save(proj_path, proj)

    pend = generate_pendulum_data()
    np.save(pend_path, pend)

proj_raw = np.load(proj_path)
pend_raw = np.load(pend_path)

print("Projectile:", proj_raw.shape, proj_raw.dtype)
print("Pendulum:", pend_raw.shape, pend_raw.dtype)

## Scaling + sanity checks

- Projectile uses **min-max to [0,1]**
- Pendulum uses **standardization (mean 0, std 1)**

In [ ]:
from src.data_generation.utils import scale_trajectories, unscale_trajectories
from src.data_generation.projectile import compute_energy_projectile
from src.data_generation.pendulum import compute_energy_pendulum

proj_scaled, proj_stats = scale_trajectories(proj_raw, mode="minmax01")
pend_scaled, pend_stats = scale_trajectories(pend_raw, mode="standardize")

print("[projectile] scaled min:", proj_scaled.reshape(-1, 4).min(axis=0))
print("[projectile] scaled max:", proj_scaled.reshape(-1, 4).max(axis=0))

print("[pendulum] scaled mean:", pend_scaled.reshape(-1, 2).mean(axis=0))
print("[pendulum] scaled std:", pend_scaled.reshape(-1, 2).std(axis=0))

# Energy conservation checks in raw space
E_proj = compute_energy_projectile(proj_raw)
E_pend = compute_energy_pendulum(pend_raw)
print("Projectile mean within-trajectory energy std:", float(E_proj.std(axis=1).mean()))
print("Pendulum mean within-trajectory energy std:", float(E_pend.std(axis=1).mean()))

## Train CDN + validate (scatter plot vs analytical energy)

In [ ]:
from src.training.train_cdn import CDNTrainConfig, train_cdn
from src.evaluation.validate_cdn import validate_cdn

# Ensure outputs go to the project folders even when executed from notebooks/
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.chdir(PROJECT_ROOT)


def run_cdn_for_env(env_name: str, raw: np.ndarray, scaled: np.ndarray, stats, state_dim: int, scaling_mode: str):
    # Mild energy alignment to avoid learning arbitrary monotone transforms (esp. pendulum)
    energy0 = None
    cfg = CDNTrainConfig(
        env_name=env_name,
        state_dim=state_dim,
        epochs=80,
        batch_size=256,
        lr=1e-3,
        lambda_var=0.1,
        epsilon=1.0,
        var_reg="hinge",
        grad_clip=1.0,
        log_grad_norm=False,
        save_dir=MODELS_DIR,
    )

    if env_name == "pendulum":
        denorm_train = unscale_trajectories(scaled, stats, mode=scaling_mode)
        energy0 = compute_energy_pendulum(denorm_train)[:, 0]
        cfg.lambda_align = 0.2

    model, _hist = train_cdn(scaled, cfg, energy0_np=energy0)

    if env_name == "projectile":
        energy_fn = lambda t: compute_energy_projectile(unscale_trajectories(t, stats, mode=scaling_mode))
    else:
        energy_fn = lambda t: compute_energy_pendulum(unscale_trajectories(t, stats, mode=scaling_mode))

    r2 = validate_cdn(model, scaled, energy_fn, env_name=env_name)
    return r2

r2_proj = run_cdn_for_env("projectile", proj_raw, proj_scaled, proj_stats, state_dim=4, scaling_mode="minmax01")
r2_pend = run_cdn_for_env("pendulum", pend_raw, pend_scaled, pend_stats, state_dim=2, scaling_mode="standardize")

print("Final R^2 projectile:", r2_proj)
print("Final R^2 pendulum:", r2_pend)

## Sparse equation recovery (Polynomial + SINDy)

Run the equation discovery script from the project root:

```powershell
python -m scripts.run_equation_discovery
```

This will:
- Fit a sparse polynomial conservation model with L1/LASSO on a normalized feature library
- Fit a SINDy-style STLSQ sparse regression baseline on the same candidate library
- Optionally run PySR (requires Julia) to confirm the functional form


In [ ]:
# This step is intentionally run as a script since it includes multiple baselines
# (Polynomial + L1/LASSO + SINDy STLSQ + optional PySR).
#
# Run from project root:
#   python -m scripts.run_equation_discovery
#
# If you want to run it from within the notebook, uncomment below.
#
# import subprocess, sys
# subprocess.check_call([sys.executable, "-u", "-m", "scripts.run_equation_discovery"])

## Outputs + optional extras

After running the notebook + equation discovery script, you should have:
- CDN validation:
  - `figures/cdn_validation_projectile.png`
  - `figures/cdn_validation_pendulum.png`
- Probing figures:
  - `figures/probe_projectile.png`
  - `figures/probe_pendulum.png`
- Equation discovery:
  - `models/poly_cdn_projectile_best.pt`, `models/poly_cdn_pendulum_best.pt`
  - `figures/equation_validation_projectile.png`, `figures/equation_validation_pendulum.png`
  - SINDy/STLSQ is printed in the terminal
  - Optional PySR Pareto fronts (requires Julia)

End-to-end script (skip-if-exists):

```powershell
python -m scripts.run_all
```
